In [ ]:
import subprocess, sys, os, glob, shutil, time, json, importlib.util
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:], flush=True); return r
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/thuong_moi.py", recursive=True) if "/notebooks/" not in p]
print(src, flush=True); assert len(src) == 1, "DUNG: can dung mot thuong_moi.py"
shutil.copytree(src[0], W, dirs_exist_ok=True)
for z in glob.glob(W + "/*.zip"): sh(f"unzip -o -q {z} -d {W}/{os.path.basename(z)[:-4]} && rm {z}")
assert os.path.isdir(f"{W}/c1/c1score"), "DUNG: thieu c1"
B = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/p1_train_rows.jsonl", recursive=True) if "/notebooks/" not in p]
print("BUNDLE", B, flush=True); assert B, "DUNG: thieu p1_train_rows.jsonl"; B = B[0]
sh(f"ls {W}; ls {W}/c1 {W}/c1/c1score | head -20")
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless >/dev/null; update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q pycocoevalcap sacrebleu bert_score 2>&1 | tail -2")
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    s = [p for p in glob.glob(f"/kaggle/input/**/{j}", recursive=True)]; assert s, f"DUNG: thieu {j}"
    shutil.copyfile(s[0], f"{SPL}/{j}"); assert os.path.getsize(f"{SPL}/{j}") == sz
print("jar SPICE du", flush=True)


In [ ]:
KQ = {}
def chay(ten, lenh):
    t0 = time.time(); log = f"{W}/{ten}.log"
    p = subprocess.Popen(f"cd {W} && {sys.executable} -u {lenh} > {log} 2>&1", shell=True)
    while p.poll() is None:
        time.sleep(60); print(time.strftime("%H:%M"), ten, open(log, errors="ignore").read()[-200:].replace("\n", " | "), flush=True)
    KQ[ten] = p.returncode; print("=====", ten, "ma thoat", p.returncode, f"{time.time()-t0:.0f} s", flush=True); print(open(log, errors="ignore").read()[-6000:], flush=True)
chay("selftest_moi", f"thuong_moi.py --selftest-moi --bundle {B}")
chay("selftest_luoi", f"grpo_luoi.py --selftest-thuong --bundle {B} --rewards spice,cider")
chay("h2_thu", f"h2_thuong.py --c1 c1 --bundle {B} --n 30 --rewards spice,spice_cong,ten_loai,loai,cider --out out/h2_thu.json")
if all(v == 0 for v in KQ.values()):
    chay("h2_thuong", f"h2_thuong.py --c1 c1 --bundle {B} --out out/h2_thuong.json")
print(KQ, flush=True)


In [ ]:
for f in glob.glob(f"{W}/out/*.json") + glob.glob(f"{W}/*.log"):
    shutil.copyfile(f, "/kaggle/working/" + os.path.basename(f))
shutil.rmtree(W); print(os.listdir("/kaggle/working"), KQ, flush=True)
assert all(v == 0 for v in KQ.values()), KQ
